In [ ]:
#!pip install biom-format
import biom
from biom import load_table
import pandas as pd
import csv
import wget
import os
from skbio import TreeNode
import tarfile

In [ ]:
# Load data from the Earth Microbiome Project (EMP) if not already present
file_Paths = ["emp_cr_silva_16S_123.release1.biom",
              "emp_cr_silva_16S_123.release1.summary.txt",
              "emp_observation_info_cr_silva.tar.gz",
              "emp_qiime_mapping_release1_20170912.tsv"]
urls = ["https://zenodo.org/records/890000/files/emp_cr_silva_16S_123.release1.biom?download=1", 
        "https://zenodo.org/records/890000/files/emp_cr_silva_16S_123.release1.summary.txt?download=1", 
        "https://zenodo.org/records/890000/files/emp_observation_info_cr_silva.tar.gz?download=1", 
        "https://zenodo.org/records/890000/files/emp_qiime_mapping_release1_20170912.tsv?download=1"]

for file, url in zip(file_Paths, urls):
    if not os.path.exists(file):
        wget.download(url, file)
        print(f'Downloaded {file}.')
    else:
        print(f"{file} already exists, skipping download.")

#Load EMP BIOM data as table
table = load_table("emp_cr_silva_16S_123.release1.biom")
# print(table.shape)

In [ ]:
# Extract tree data and plot an ASCII tree graph

with tarfile.open("emp_observation_info_cr_silva.tar.gz", "r:*") as tar:
    tree_file = "silva_123.97_otus.tre"
    tree = TreeNode.read(tar.extractfile(tree_file))
    tar.extract(tree_file, path=".")

print(tree.ascii_art())

# ipx.plotting.tree(
#     tree,
#     layout="radial",
#     aspect=1,
#     edge_color="purple",
# )

In [ ]:
# Load EMP Metadata TSV as a pandas dataframe and filter for forests and soils
emp_metadata = pd.read_csv("emp_qiime_mapping_release1_20170912.tsv", sep = "\t")
emp_metadata_filtered = emp_metadata[emp_metadata["sample_scientific_name"].str.contains("soil metagenome") & emp_metadata["envo_biome_2"].str.contains("forest biome") ]
emp_metadata_filtered = emp_metadata_filtered[~emp_metadata_filtered["Description"].str.contains("Boreal") & ~emp_metadata_filtered["env_feature"].str.contains("taiga")] 
#print(emp_metadata_filtered["env_biome"].unique())

# Then split the data into a tropical and temperate dataframe
tropical_emp = emp_metadata_filtered[emp_metadata_filtered["env_biome"].str.contains("tropical")]
temperate_emp = emp_metadata_filtered[~emp_metadata_filtered["env_biome"].str.contains("tropical")]

# Get the sample ids from the metadata, apply the filter to EMP table, and delete taxa that are zero
table_ids = set(table.ids(axis="sample"))
sample_ids = set(emp_metadata_filtered["#SampleID"].values)
valid_ids = table_ids & sample_ids

tropical_ids = valid_ids & set(tropical_emp["#SampleID"].values)
temperate_ids = valid_ids & set(temperate_emp["#SampleID"].values)
tropical_emp[tropical_emp["#SampleID"].isin(tropical_ids)].to_csv('tropical_metadata.tsv', sep="\t")
temperate_emp[temperate_emp["#SampleID"].isin(temperate_ids)].to_csv('temperate_metadata.tsv', sep="\t")

filtered_table = table.filter(
    ids_to_keep= valid_ids,
    axis="sample",
    inplace=False
)

filtered_table = filtered_table.filter(
    ids_to_keep= lambda values, id_, metadata: values.sum() > 0,
    axis="observation",
    inplace=False
)

tropical = filtered_table.filter(
    ids_to_keep= tropical_ids,
    axis="sample",
    inplace=False
)

temperate = filtered_table.filter(
    ids_to_keep= temperate_ids,
    axis="sample",
    inplace=False
)

with open("tropical.tsv", "w") as f:
    f.write(tropical.to_tsv())

with open("temperate.tsv", "w") as f:
    f.write(temperate.to_tsv())

taxonomy = {oid: filtered_table.metadata(oid, axis="observation") for oid in filtered_table.ids(axis="observation")}

header = ["#OTU_ID", "Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]

# Writing the dictionary to CSV
with open("taxonomy.tsv", mode='w', newline='') as file:
    writer = csv.writer(file, delimiter='\t', lineterminator='\n')
    writer.writerow(header)
    for key,values in taxonomy.items():
        data = [key]
        for item in values["taxonomy"]:
            data.append(item.split("__")[1])
        writer.writerow(data)

del emp_metadata_filtered
del tropical_emp
del temperate_emp
del emp_metadata

# print(tropical.shape)
# print(temperate.shape)